# Legacy replication — was the submission's BaCP right?

Re-runs the **submission's BaCP configuration** on the FIXED infrastructure (honest eval over all 10,000 test images, working data pipeline, recorded provenance) for ResNet-50 / CIFAR-10, **BaCP + magnitude only**:

| knob | value | why |
|---|---|---|
| `contrastive_mode` | `legacy` | the original 2B×2B SupCon+NTXent composite over `cat([student, teacher])` |
| `proj_mode` | `current` | per-model projection heads, student's trainable (original behavior) |
| `tau` | 0.07 | the submission's operative temperature (parser default) |
| regime | 5 epochs + 10 interleaved recovery, ΔT=100 | the submission's schedule |
| head | dense (`prune_task_head` off) | the submission's scope |

Records carry a **`.legacy`** key suffix, so they coexist with the main table instead of overwriting it.

**Reading the verdict** against the submission's BaCP+magnitude (93.58 / 93.27 / 92.32):
- **Reproduces (±~0.4)** → the old numbers were real: the design (SimCLR-composite objective + trainable head) genuinely produced them, and every old-vs-new delta is attributable to methodology changes, not measurement error.
- **Falls short** → the old numbers carried measurement artifacts (`drop_last` eval scored 9,728 batch-dependent images) and/or luck; the gap quantifies it.


In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Run: BaCP + magnitude × {0.95, 0.97, 0.99}, submission config

Uses the existing dense checkpoint (seed 1). ~25 min per sparsity on the A100.

In [ ]:
GPU = 0
SEED = 1

LEGACY = dict(contrastive_mode='legacy', proj_mode='current', tau=0.07,
              epochs=5, recovery_epochs=10, delta_T=100,
              prune_task_head=False)

cells = [nb.make_cell('resnet50', 'bacp', seed=SEED, pruner='magnitude',
                      sparsity=s, variant='legacy', **LEGACY)
         for s in (0.95, 0.97, 0.99)]
nb.run_group(cells, gpu=GPU)

## Verdict

In [ ]:
import json, glob, os
OLD = {0.95: 93.58, 0.97: 93.27, 0.99: 92.32}
root = os.environ['BACP_RESULTS_DIR']
print(f'{"sparsity":<9} {"legacy (fixed infra)":<21} {"submission":<11} {"delta":<7}')
for s in (0.95, 0.97, 0.99):
    acc = None
    for f in glob.glob(os.path.join(root, 'runs', '*.json')):
        rec = json.load(open(f, encoding='utf-8'))
        if rec.get('experiment_group') == f'static.bacp.resnet50.cifar10.s{s}.magnitude.seed1.legacy':
            acc = rec.get('test_acc_pct')
    shown = f'{acc:.2f}' if acc is not None else 'not run'
    delta = f'{acc - OLD[s]:+.2f}' if acc is not None else '-'
    print(f'{s:<9} {shown:<21} {OLD[s]:<11} {delta:<7}')
print()
print('Reproduces -> old numbers were real (deltas = methodology).')
print('Falls short -> old numbers carried the eval defect / noise.')